In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np 
from  sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
import seaborn as sns
import optuna
import warnings
warnings.filterwarnings("ignore")

In [34]:
DATA_DIR = Path.cwd().parent / "data"
df = pd.read_csv(DATA_DIR / "processed" / "cleaned_taxi_data.csv")
df.head(5)

,index_left,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,BoroName
0,4217,id1693416,2,2016-01-12 22:57:13,2016-01-12 23:33:06,1,-73.979324,40.727631,-74.074028,40.640224,N,35.883333,Staten Island
1,16494,id1973056,2,2016-03-22 15:35:52,2016-03-22 16:53:17,1,-73.789642,40.647121,-74.135986,40.624981,N,77.416667,Staten Island
2,17985,id3759847,2,2016-02-04 22:00:26,2016-02-04 22:48:18,2,-73.984749,40.742130,-74.076393,40.599159,N,47.866667,Staten Island
3,19015,id3330882,2,2016-04-02 08:47:43,2016-04-02 08:50:23,1,-74.073433,40.615421,-74.073433,40.615421,N,2.666667,Staten Island
4,19701,id0778469,1,2016-01-01 19:46:50,2016-01-01 20:33:26,1,-73.991974,40.749928,-74.115303,40.574230,N,46.600000,Staten Island


In [35]:
def haversine_form(lat1,long1,lat2,long2):
    lat1 , long1,lat2,long2 = map(np.radians, [lat1, long1, lat2, long2])
    dlat = lat2-lat1
    dlong = long2-long1
    a  = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2)*np.sin(dlong/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 3956
    return c*r

In [36]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df.drop(['vendor_id','dropoff_datetime','store_and_fwd_flag'],axis =1,inplace=True)
df['distance'] =  haversine_form(df['pickup_latitude'],df['pickup_longitude'],df['dropoff_latitude'],df['dropoff_longitude'])
df['day_of_week'] = df['pickup_datetime'].dt.day_of_week
df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['is_rush_hour'] = df['pickup_hour'].isin([7,8,9,16,17,18,19]).astype(int)
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

In [37]:
X = df.drop(['trip_duration','id','pickup_datetime','index_left'],axis=1)
y = df['trip_duration']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,shuffle=True)

In [38]:
categorical = X.select_dtypes(include=['str']).columns.to_list()
X[categorical]= X[categorical].astype('category')

In [48]:

def objective(trial):
    param = {
        "loss":'absolute_error',
        "learning_rate": trial.suggest_float("learning_rate",0.001,1.0),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.001, 10.0),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes",2,100),
        "max_depth" :trial.suggest_int("max_depth",1,10),
        "min_samples_leaf" :trial.suggest_int("min_samples_leaf",2,50)
    }

    model = HistGradientBoostingRegressor(**param,categorical_features=categorical,random_state=42)
    model.fit(X_train.iloc[:50000],y_train.iloc[:50000])
    prediction = model.predict(X_test.iloc[:20000])

    return mean_absolute_error(y_test.iloc[:20000],prediction)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(storage="sqlite:///db.sqlite3",study_name="hist_optuna",direction='minimize',load_if_exists=True)
study.optimize(objective, n_trials=100,show_progress_bar=True)
print(study.best_params)
print(study.best_value)



Best trial: 82. Best value: 3.10021: 100%|██████████| 100/100 [07:12<00:00,  4.33s/it]

{'learning_rate': 0.20822254137715343, 'l2_regularization': 1.5234018422544593, 'max_leaf_nodes': 100, 'max_depth': 10, 'min_samples_leaf': 33}
3.100209198892046


In [49]:
model = HistGradientBoostingRegressor(**study.best_params,categorical_features=categorical,random_state=42)
model.fit(X_train,y_train)
prediction = model.predict(X_test)
print(f"Baseline Mean Absolute Error {mean_absolute_error(y_test,prediction):.2f}")
print(f"R2 {r2_score(y_test,prediction):.2f}")


Baseline Mean Absolute Error 3.08
R2 0.80


In [50]:
train_pred = model.predict(X_train)
print(f"Train MAE: {mean_absolute_error(y_train, train_pred):.2f}")
print(f"Test MAE: {mean_absolute_error(y_test, prediction):.2f}")

Train MAE: 3.02
Test MAE: 3.08
